# Módulo 1 — EDA: Predicción de Demanda de Transporte
**Dataset real**: `amanmehra23/travel-recommendation-dataset` (Kaggle)

Universidad Nacional de Colombia · IRNA 2026-01

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

OUTPUT_DIR = Path('.')
OUTPUT_DIR.mkdir(exist_ok=True)
print("Librerías cargadas.")

## 1. Descarga y carga del dataset real de Kaggle

In [ ]:
import kagglehub, sys

print("Descargando dataset desde Kaggle...")
try:
    path = Path(kagglehub.dataset_download("amanmehra23/travel-recommendation-dataset"))
    print(f"Path: {path}")
except Exception as e:
    sys.exit(f"ERROR: No se pudo descargar el dataset.\n{e}\n"
             "Configura ~/.kaggle/kaggle.json con tus credenciales.")

csvs = {p.stem.lower(): p for p in path.rglob("*.csv")}
print("CSVs disponibles:", list(csvs.keys()))

reviews_path = next((p for k,p in csvs.items() if "review" in k), None)
dest_path    = next((p for k,p in csvs.items() if "destination" in k), None)
if not reviews_path or not dest_path:
    sys.exit(f"No se encontraron reviews o destinations. CSVs: {list(csvs.values())}")

df_reviews = pd.read_csv(reviews_path)
df_dest    = pd.read_csv(dest_path)
print(f"Reviews: {df_reviews.shape}  |  Destinations: {df_dest.shape}")
print(df_reviews.head(3))

## 2. Exploración inicial

In [ ]:
print("=== Reviews ===")
print(df_reviews.dtypes)
print(df_reviews.describe())
print("\n=== Destinations ===")
print(df_dest.dtypes)
print(df_dest.describe())

## 3. Columnas de fecha y destino

In [ ]:
# Detectar columna de fecha
date_col = None
for col in df_reviews.columns:
    try:
        pd.to_datetime(df_reviews[col].dropna().head(10), errors='raise')
        date_col = col; break
    except: pass
print(f"Columna de fecha: {date_col}")

df_reviews[date_col] = pd.to_datetime(df_reviews[date_col], errors='coerce')
df_reviews = df_reviews.dropna(subset=[date_col])
print(f"Rango: {df_reviews[date_col].min()} → {df_reviews[date_col].max()}")
print(f"Reviews con fecha válida: {len(df_reviews)}")

## 4. Construcción de series temporales por destino

In [ ]:
# Join para obtener Name del destino
if "DestinationID" in df_reviews.columns and "Name" in df_dest.columns:
    df_reviews = df_reviews.merge(
        df_dest[["DestinationID","Name"]].drop_duplicates("DestinationID"),
        on="DestinationID", how="left"
    )
    dest_col = "Name"
else:
    dest_col = next(c for c in df_reviews.columns if df_reviews[c].dtype==object)

print(f"Columna destino: {dest_col}")
df_reviews["date_only"] = df_reviews[date_col].dt.normalize()
daily = df_reviews.groupby([dest_col,"date_only"]).size().reset_index(name="demand")

dest_days = daily.groupby(dest_col)["date_only"].nunique()
top5 = dest_days.sort_values(ascending=False).head(5).index.tolist()
print(f"Top 5 destinos por días con datos: {top5}")

## 5. Visualización de series temporales (Top 5)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 14))
for ax, dest in zip(axes, top5):
    sub = daily[daily[dest_col]==dest].set_index("date_only")["demand"].sort_index()
    full_idx = pd.date_range(sub.index.min(), sub.index.max(), freq="D")
    sub = sub.reindex(full_idx).interpolate("linear").ffill().bfill()
    ax.plot(sub.index, sub.values, linewidth=0.8, color="#2C5282")
    ax.fill_between(sub.index, sub.values, alpha=0.15, color="#2C5282")
    rm = sub.rolling(30, min_periods=1).mean()
    ax.plot(rm.index, rm.values, color="#E53E3E", linewidth=2, label="Media móvil 30d")
    ax.set_title(dest, fontsize=12, fontweight='bold')
    ax.set_ylabel("Demanda diaria")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
plt.suptitle("Series Temporales de Demanda por Destino (datos reales)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_series_temporales.png", dpi=120, bbox_inches='tight')
plt.show()
print("Guardado: fig_series_temporales.png")

## 6. Análisis de estacionalidad por mes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_reviews["month"] = df_reviews[date_col].dt.month
monthly = df_reviews.groupby("month").size()
axes[0].bar(monthly.index, monthly.values, color="#3182CE")
axes[0].set_xlabel("Mes"); axes[0].set_ylabel("Reviews"); axes[0].set_title("Demanda por Mes")
axes[0].set_xticks(range(1,13))
axes[0].set_xticklabels(["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"])

df_reviews["dow"] = df_reviews[date_col].dt.dayofweek
dow = df_reviews.groupby("dow").size()
axes[1].bar(dow.index, dow.values, color="#48BB78")
axes[1].set_xlabel("Día"); axes[1].set_ylabel("Reviews"); axes[1].set_title("Demanda por Día de Semana")
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(["Lun","Mar","Mié","Jue","Vie","Sáb","Dom"])
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_estacionalidad.png", dpi=120, bbox_inches='tight')
plt.show()

## 7. Top 15 destinos más populares

In [ ]:
top15 = daily.groupby(dest_col)["demand"].sum().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(12,6))
ax.barh(top15.index[::-1], top15.values[::-1], color="#3182CE")
ax.set_xlabel("Demanda total acumulada"); ax.set_title("Top 15 Destinos por Demanda Total")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_top15_destinos.png", dpi=120, bbox_inches='tight')
plt.show()
print("Top 5:", top5)

## 8. Análisis de valores faltantes y calidad

In [ ]:
print("=== Valores faltantes en reviews ===")
miss = df_reviews.isnull().sum()
print(miss[miss>0] if miss.any() else "Sin valores faltantes")
print(f"\nTotal registros: {len(df_reviews)}")
print(f"Destinos únicos: {df_reviews[dest_col].nunique()}")
if 'Rating' in df_reviews.columns:
    print(f"Rating medio: {df_reviews['Rating'].mean():.2f}")
    df_reviews['Rating'].hist(bins=10, color='#3182CE', edgecolor='white', figsize=(8,4))
    plt.title("Distribución de Ratings"); plt.xlabel("Rating"); plt.ylabel("Frecuencia")
    plt.savefig(OUTPUT_DIR / "fig_rating_dist.png", dpi=120, bbox_inches='tight')
    plt.show()

## 9. Conclusiones del EDA
- Dataset real de Kaggle con datos de múltiples destinos.
- Se identificaron los top 5 destinos con suficiente historia temporal.
- Estacionalidad mensual y semanal visible en las series.
- Datos aptos para modelar con LSTM (series temporales diarias).